# Summary_Day1.ipynb  
## 딥러닝 필수 파이썬, 실습 환경구성 및 OOP

이번 노트북은 **텍스트 설명 셀**과 **실행 가능한 코드 셀**을 분리해서 정리했습니다.

학습 흐름:

1. 딥러닝 학습 구조 이해
2. Loss 개념 이해
3. Optimizer와 업데이트 공식 이해
4. 컨테이너 타입과 `copy()` 이해
5. 합성 함수 이해
6. 수치 미분 구현
7. PyTorch Autograd 이해
8. Class / OOP 이해
9. `__call__` 구조 이해
10. PyTorch `nn.Module` 구조로 연결

## 1. 딥러닝 학습 흐름

딥러닝 학습은 사람이 문제를 풀고, 채점하고, 틀린 이유를 보고, 다시 고치는 과정과 비슷합니다.

딥러닝에서는 이 흐름을 다음과 같이 말합니다.

```text
Forward → Loss → Backward → Update
```

- `Forward`: 입력 데이터를 보고 예측합니다.
- `Loss`: 예측값과 정답을 비교합니다.
- `Backward`: 오차의 원인을 거꾸로 계산합니다.
- `Update`: 파라미터를 수정합니다.

In [ ]:
steps = [
    "1. Forward: 입력 데이터를 보고 예측한다",
    "2. Loss: 예측값과 정답을 비교한다",
    "3. Backward: 오차의 원인을 거꾸로 계산한다",
    "4. Update: 파라미터를 수정한다",
]

for step in steps:
    print(step)

## 2. Loss 개념

`Loss`는 모델이 얼마나 틀렸는지를 숫자로 표현한 값입니다.

예를 들어 정답이 `5`, 예측값이 `4`라면 차이는 `1`입니다.

Loss가 작을수록 모델이 정답에 가까워진 것입니다.

In [ ]:
target = 5
prediction = 4

loss = target - prediction

print("정답:", target)
print("예측:", prediction)
print("손실:", loss)

## 3. 파라미터 업데이트 공식

딥러닝은 파라미터를 조금씩 수정하면서 Loss를 줄입니다.

핵심 공식은 다음과 같습니다.

```text
Param = Param - (lr * grad)
```

- `Param`: 수정할 파라미터
- `lr`: Learning Rate, 학습률
- `grad`: Gradient, 기울기

In [ ]:
param = 10.0
lr = 0.1
grad = 2.0

new_param = param - (lr * grad)

print("수정 전 param:", param)
print("수정 후 param:", new_param)

## 4. 컨테이너 타입과 얕은 복사

리스트, 딕셔너리, NumPy 배열은 데이터를 담는 그릇입니다.

그런데 `y = x`처럼 대입하면 값이 새로 복사되는 것이 아니라, 같은 데이터를 함께 바라보게 됩니다.

이를 **얕은 복사** 또는 **참조 복사**라고 이해하면 됩니다.

In [ ]:
x = [5, 7, 9]

y = x

x[0] = 100

print("x:", x)
print("y:", y)

위 코드에서는 `x[0]`만 바꿨는데 `y`도 함께 바뀝니다.

이유는 `x`와 `y`가 같은 리스트를 가리키고 있기 때문입니다.

데이터 전처리 과정에서는 이런 문제가 원본 데이터 훼손으로 이어질 수 있습니다.

## 5. `copy()`로 안전하게 복사하기

원본 데이터를 보호하려면 `copy()`를 사용합니다.

`copy()`를 사용하면 새로운 메모리 공간에 복사본이 만들어집니다.

In [ ]:
x = [5, 7, 9]

y = x.copy()

x[0] = 100

print("x:", x)
print("y:", y)

이번에는 `x`를 수정해도 `y`는 바뀌지 않습니다.

즉, `copy()`를 사용하면 원본과 복사본을 따로 관리할 수 있습니다.

## 6. NumPy 배열에서 `copy()` 사용하기

딥러닝과 데이터 분석에서는 NumPy 배열을 자주 사용합니다.

NumPy 배열도 참조 문제가 생길 수 있기 때문에, 원본을 보존해야 한다면 `copy()`를 사용해야 합니다.

In [ ]:
import numpy as np

x = np.array([5, 7, 9])

y = x.copy()

x[0] = 100

print("x:", x)
print("y:", y)

## 7. 합성 함수 이해하기

딥러닝 모델은 여러 함수가 순서대로 연결된 **거대한 합성 함수**라고 볼 수 있습니다.

```text
y = f3(f2(f1(x)))
```

입력값이 여러 함수 블록을 지나면서 최종 결과로 변환됩니다.

In [ ]:
def f1(x):
    return x ** 2

def f2(x):
    return 2 * x

def f3(x):
    return x + 2

x = 3

y = f3(f2(f1(x)))

print(y)

코드 흐름은 다음과 같습니다.

```text
x = 3
f1(3) = 9
f2(9) = 18
f3(18) = 20
```

그래서 최종 출력은 `20`입니다.

딥러닝의 Layer 구조도 이와 비슷하게 이해하면 됩니다.

## 8. 합성 함수 과정을 나눠서 확인하기

초보자는 합성 함수를 한 줄로 보는 것보다, 중간 결과를 변수에 저장해서 확인하는 것이 좋습니다.

In [ ]:
x = 3

a = f1(x)
b = f2(a)
c = f3(b)

print("처음 x:", x)
print("f1 결과:", a)
print("f2 결과:", b)
print("f3 결과:", c)

## 9. 미분의 역할

강의에서 미분은 다음 질문으로 설명할 수 있습니다.

```text
어느 방향으로 고쳐야 하는가?
```

미분은 Loss를 줄이기 위해 파라미터를 어느 방향으로 얼마나 바꿔야 하는지 알려줍니다.

## 10. 중앙 차분 수치 미분

중앙 차분은 `x`를 기준으로 양쪽의 아주 가까운 값을 사용해서 기울기를 근사하는 방법입니다.

```text
f'(x) ≈ (f(x+h) - f(x-h)) / 2h
```

In [ ]:
def fdiff(f):
    def diff(x):
        h = 1e-6
        return (f(x + h) - f(x - h)) / (2 * h)
    return diff

`fdiff(f)`는 함수를 입력받아서 그 함수의 미분 함수를 만들어줍니다.

여기서 `h = 1e-6`은 아주 작은 값입니다.

## 11. 수치 미분 테스트

다음 함수를 미분해봅니다.

```text
f(x) = 2x² + 2
```

실제 미분은 다음과 같습니다.

```text
f'(x) = 4x
```

따라서 `x = 3`일 때 미분값은 `12`가 되어야 합니다.

In [ ]:
f = lambda x: 2 * x**2 + 2

f_prime = fdiff(f)

print(f_prime(3.0))

출력값이 `12`에 가깝게 나오면 수치 미분이 잘 동작한 것입니다.

실제 딥러닝에서는 이런 계산을 직접 구현하지 않고 PyTorch의 Autograd가 자동으로 처리합니다.

## 12. PyTorch Autograd

PyTorch의 `Autograd`는 자동 미분 기능입니다.

`requires_grad=True`를 설정하면 PyTorch가 해당 값의 계산 과정을 추적합니다.

In [ ]:
import torch

x = torch.tensor(3.0, requires_grad=True)

y = 2 * x**2 + 2

y.backward()

print(x.grad)

결과는 `tensor(12.)`입니다.

앞에서 직접 계산한 수치 미분 결과와 같습니다.

즉, PyTorch가 자동으로 미분값을 계산해준 것입니다.

## 13. Class 개념

PyTorch 모델은 보통 클래스로 만듭니다.

클래스는 객체를 만들기 위한 설계도입니다.

| 개념 | 의미 |
|---|---|
| Class | 설계도 |
| Instance | 설계도로 만든 실제 객체 |
| Attribute | 객체가 가진 데이터 |
| Method | 객체가 수행하는 기능 |

In [ ]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def move(self, dx, dy):
        self.x += dx
        self.y += dy

    def coord(self):
        return (self.x, self.y)

`__init__`은 객체가 만들어질 때 처음 실행되는 생성자입니다.

`self.x`, `self.y`는 객체 내부에 저장되는 속성입니다.

`move()`와 `coord()`는 객체가 수행할 수 있는 메서드입니다.

## 14. Point 클래스 사용하기

이제 Point 클래스로 실제 객체를 만들고 좌표를 이동시킵니다.

In [ ]:
p = Point(2, 3)

print("처음 좌표:", p.coord())

p.move(1, -1)

print("이동 후 좌표:", p.coord())

처음 좌표는 `(2, 3)`입니다.

`move(1, -1)`을 실행하면 다음처럼 바뀝니다.

```text
x = 2 + 1 = 3
y = 3 - 1 = 2
```

그래서 결과는 `(3, 2)`입니다.

## 15. `__call__` 이해하기

클래스 안에 `__call__` 메서드를 만들면 객체를 함수처럼 사용할 수 있습니다.

이 개념은 PyTorch 모델을 이해할 때 중요합니다.

PyTorch에서는 보통 다음처럼 모델 객체를 함수처럼 호출합니다.

```python
model(x)
```

In [ ]:
class H:
    def __call__(self, x):
        return 2 * x + 2

h = H()

result = h(3)

print(result)

`h(3)`을 실행하면 내부적으로 `h.__call__(3)`이 실행됩니다.

그래서 결과는 다음과 같습니다.

```text
2 * 3 + 2 = 8
```

PyTorch의 `nn.Module`도 이 구조를 사용하기 때문에 `model.forward(x)`보다 `model(x)` 형태로 사용합니다.

## 16. PyTorch 모델 구조 예고

PyTorch 모델은 보통 다음 구조를 가집니다.

```python
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Layer 준비

    def forward(self, x):
        # 순전파 정의
        return x
```

이번 차시에서 배운 함수, 합성 함수, 클래스, `__call__` 개념이 모두 이 구조로 연결됩니다.

In [ ]:
import torch
import torch.nn as nn

class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Linear(1, 1)

    def forward(self, x):
        return self.layer(x)

model = MyModel()

sample_x = torch.tensor([[1.0]])

output = model(sample_x)

print(output)

이 코드는 실제 학습을 하는 코드는 아니고, PyTorch 모델 구조를 미리 보는 예시입니다.

핵심은 다음과 같습니다.

- `nn.Module`을 상속받습니다.
- `__init__`에서 Layer를 준비합니다.
- `forward()`에서 데이터 흐름을 정의합니다.
- 사용할 때는 `model(x)`처럼 호출합니다.

## 17. 주요 약어 정리

| 약어 | 의미 | 설명 |
|---|---|---|
| `np` | NumPy | 수치 계산 라이브러리 |
| `plt` | matplotlib.pyplot | 그래프 그리는 도구 |
| `torch` | PyTorch | 딥러닝 라이브러리 |
| `nn` | Neural Network | 신경망 모듈 |
| `optim` | Optimizer | 최적화 도구 |
| `X` | Input | 입력값 |
| `y` | Target | 정답값 |
| `pred` | Prediction | 예측값 |
| `loss` | Loss | 오차 |
| `grad` | Gradient | 기울기 |
| `lr` | Learning Rate | 학습률 |
| `epoch` | Epoch | 전체 데이터를 반복 학습하는 단위 |

## 18. 시험용 요약

```text
딥러닝 학습 = Forward → Loss → Backward → Update 반복
```

- Loss는 모델이 얼마나 틀렸는지를 나타냅니다.
- Gradient는 어느 방향으로 수정해야 하는지 알려줍니다.
- Optimizer는 gradient를 이용해 파라미터를 수정합니다.
- 딥러닝 모델은 여러 함수가 연결된 합성 함수 구조입니다.
- 수치 미분은 미분 원리를 이해하기 위한 방법입니다.
- 실제 딥러닝에서는 PyTorch Autograd가 자동 미분을 수행합니다.
- Class는 데이터와 기능을 하나로 묶는 구조입니다.
- `__call__` 덕분에 객체를 함수처럼 호출할 수 있습니다.
- PyTorch 모델은 `nn.Module`을 상속받는 클래스 구조입니다.
- 컨테이너 타입은 참조 문제 때문에 `copy()`를 사용해야 할 때가 있습니다.